# 03 Generate Final Report with Qwen7B

This notebook generates a Markdown report for the CarDD YOLO11 segmentation reproduction project.

Dependency order:

1. Run **Mount Drive**.
2. Run **Install and Import**.
3. Run **Configure Paths**.
4. Run **Build Report Context** after notebooks 01 and 02 have completed.
5. Run **Load Qwen7B** on a GPU runtime.
6. Run **Generate and Save Report**.

The generated report is saved to Drive. Qwen model weights are not saved in this GitHub repository.

## Mount Drive

Dependency: none. Run this first in every Colab session.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Install and Import

Dependency: Drive is mounted. This installs Qwen inference dependencies.

In [ ]:
%pip install -q transformers>=4.51.0 accelerate bitsandbytes pandas

In [ ]:
import json
from pathlib import Path

import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

## Configure Paths

Dependency: imports are ready. Change only these variables if your Drive layout differs.

In [ ]:
DRIVE_ROOT = Path('/content/drive/MyDrive/CarDD_YOLO11')
REPORTS_ROOT = DRIVE_ROOT / 'reports'
TRAIN_ROOT = DRIVE_ROOT / 'runs' / 'train' / 'yolo11n_seg'
VAL_ROOT = DRIVE_ROOT / 'runs' / 'val' / 'yolo11n_seg_test'
DEMO_ROOT = DRIVE_ROOT / 'runs' / 'predict' / 'demo'

TEST_METRICS_JSON = REPORTS_ROOT / 'yolo11n_seg_test_metrics.json'
RUN_SUMMARY_JSON = REPORTS_ROOT / 'yolo11n_seg_run_summary.json'
RESULTS_CSV = TRAIN_ROOT / 'results.csv'

CONTEXT_JSON = REPORTS_ROOT / 'qwen7b_report_context.json'
REPORT_MD = REPORTS_ROOT / 'qwen7b_final_report.md'

MODEL_ID = 'Qwen/Qwen2.5-7B-Instruct'
USE_4BIT = True
REPORT_LANGUAGE = 'English'
MAX_NEW_TOKENS = 2200

REPORTS_ROOT.mkdir(parents=True, exist_ok=True)
print('Drive root:', DRIVE_ROOT)
print('Report output:', REPORT_MD)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## Build Report Context

Dependency: notebooks 01 and 02 have completed, and their outputs exist in Drive.

This cell collects metrics, training history, saved artifacts, and demo file names into one JSON context for the LLM.

In [ ]:
assert TEST_METRICS_JSON.exists(), f'Missing test metrics: {TEST_METRICS_JSON}'
assert RUN_SUMMARY_JSON.exists(), f'Missing run summary: {RUN_SUMMARY_JSON}'
assert RESULTS_CSV.exists(), f'Missing results CSV: {RESULTS_CSV}'

test_metrics = json.loads(TEST_METRICS_JSON.read_text(encoding='utf-8'))
run_summary = json.loads(RUN_SUMMARY_JSON.read_text(encoding='utf-8'))
results_df = pd.read_csv(RESULTS_CSV)
last_row = results_df.iloc[-1].to_dict()
best_box_row = results_df.iloc[results_df['metrics/mAP50(B)'].idxmax()].to_dict()
best_mask_row = results_df.iloc[results_df['metrics/mAP50(M)'].idxmax()].to_dict()

demo_images = sorted([p.name for p in DEMO_ROOT.glob('*.jpg')]) if DEMO_ROOT.exists() else []
demo_labels = sorted([p.name for p in (DEMO_ROOT / 'labels').glob('*.txt')]) if (DEMO_ROOT / 'labels').exists() else []

per_class_test_map50 = [
    {'class': 'dent', 'box_map50': 0.492, 'mask_map50': 0.496},
    {'class': 'scratch', 'box_map50': 0.511, 'mask_map50': 0.475},
    {'class': 'crack', 'box_map50': 0.211, 'mask_map50': 0.219},
    {'class': 'glass shatter', 'box_map50': 0.978, 'mask_map50': 0.978},
    {'class': 'lamp broken', 'box_map50': 0.790, 'mask_map50': 0.778},
    {'class': 'tire flat', 'box_map50': 0.882, 'mask_map50': 0.882},
]

demo_summary = [
    'image0: 1 scratch',
    'image1: no detections',
    'image2: 1 lamp broken',
    'image3: 1 dent, 1 scratch',
    'image4: 1 scratch',
    'image5: 2 cracks',
    'image6: 1 scratch',
    'image7: no detections',
]

context = {
    'project': 'CarDD YOLO11 Segmentation Reproduction',
    'dataset': 'CarDD car damage detection and instance segmentation dataset',
    'task': 'car damage detection and instance segmentation',
    'model': 'YOLO11n-seg',
    'llm_report_model': MODEL_ID,
    'training_setup': {
        'epochs': 100,
        'imgsz': 1024,
        'batch': 7,
        'gpu': 'Colab L4',
        'training_time_hours': 4.109,
        'optimizer': 'AdamW',
        'augmentations': ['multi_scale', 'mosaic', 'mixup', 'copy_paste'],
    },
    'test_metrics': test_metrics,
    'last_epoch_metrics': last_row,
    'best_box_epoch': best_box_row,
    'best_mask_epoch': best_mask_row,
    'per_class_test_map50': per_class_test_map50,
    'demo_summary': demo_summary,
    'demo_images': demo_images,
    'demo_labels': demo_labels,
    'artifacts': {
        'best_pt': run_summary.get('best_pt'),
        'last_pt': run_summary.get('last_pt'),
        'onnx': str(DRIVE_ROOT / 'exports' / 'best.onnx'),
        'test_metrics_json': str(TEST_METRICS_JSON),
        'demo_dir': str(DEMO_ROOT),
    },
}

CONTEXT_JSON.write_text(json.dumps(context, indent=2), encoding='utf-8')
print('Context saved:', CONTEXT_JSON)
print(json.dumps(context['test_metrics'], indent=2))

## Load Qwen7B

Dependency: report context exists. Use a GPU runtime. The default 4-bit path is intended for Colab L4.

If this cell fails due to dependency or CUDA memory issues, restart the runtime and rerun from the top.

In [ ]:
assert CONTEXT_JSON.exists(), 'Run Build Report Context first.'

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

if USE_4BIT:
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        device_map='auto',
        torch_dtype=torch.bfloat16,
        quantization_config=quantization_config,
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        device_map='auto',
        torch_dtype='auto',
    )

model.eval()
print('Loaded:', MODEL_ID)
if hasattr(model, 'get_memory_footprint'):
    print('Memory footprint GB:', round(model.get_memory_footprint() / 1024**3, 2))

## Generate and Save Report

Dependency: Qwen7B is loaded. This cell generates the final Markdown report and writes it to Drive.

In [ ]:
context = json.loads(CONTEXT_JSON.read_text(encoding='utf-8'))

system_prompt = (
    'You are a careful machine learning project report writer. '
    'Use only the provided context. Do not invent metrics, file paths, or claims. '
    'Write a concise but complete Markdown report for a computer vision reproduction project.'
)

user_prompt = f"""
Write the final project report in {REPORT_LANGUAGE}.

Required sections:
1. Title
2. Project Overview
3. Dataset and Task
4. Method
5. Training Setup
6. Results
7. Per-Class Analysis
8. Demo Inference
9. Saved Artifacts
10. Limitations and Future Work

Style requirements:
- Be factual and concise.
- Keep numeric metrics to three decimals where appropriate.
- Mention that large files are stored in Google Drive, not GitHub.
- Do not claim state-of-the-art performance.
- Do not include private personal information.

Context JSON:
```json
{json.dumps(context, indent=2)}
```
"""

messages = [
    {'role': 'system', 'content': system_prompt},
    {'role': 'user', 'content': user_prompt},
]

text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer([text], return_tensors='pt').to(model.device)

with torch.no_grad():
    generated_ids = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        temperature=None,
        top_p=None,
    )

generated_ids = generated_ids[:, inputs.input_ids.shape[-1]:]
report = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()

REPORT_MD.write_text(report + '\n', encoding='utf-8')
print('Report saved:', REPORT_MD)
print(report[:3000])

## Inspect Saved Report

Dependency: report generation has completed. This cell prints the generated report from Drive.

In [ ]:
assert REPORT_MD.exists(), 'Run Generate and Save Report first.'
print(REPORT_MD.read_text(encoding='utf-8'))